# SQL 生成器

读取 participants 目录下的 `user_id.json`，生成查询所有参赛者宽币赠送情况的 SQL，
输出到 sql 目录。路径统一走 `common.paths`（system/files/scripts 下）。

In [ ]:
import json
import sys
from pathlib import Path

# 笔记本工作目录是 grants/，其父目录 scripts/ 下有 common 包
sys.path.insert(0, str(Path.cwd().parent))
from common.paths import PARTICIPANTS_DIR, SQL_DIR

with open(PARTICIPANTS_DIR / "user_id.json", "r", encoding="utf-8") as f:
    user_ids = json.load(f)

in_list = ", ".join(f"'{uid}'" for uid in user_ids)
sql = (
    f"SELECT * FROM kbb__orders "
    f"WHERE user_id IN ({in_list}) "
    f"AND space_id = '00000000-0000-0000-0000-000000000000' "
    f"AND created_at > '2026-06-25' "
    f"AND type in ('bigquant_charge', 'reward') "
    f"AND JSON_LENGTH(notes) > 0;"
)

SQL_DIR.mkdir(parents=True, exist_ok=True)
out_path = SQL_DIR / "bigquant_charge.sql"
out_path.write_text(sql + "\n", encoding="utf-8")

print(f"已写入: {out_path.resolve()}")
print(sql)